# Seed `frontier_labs` Delta tables

Seeds the frontier-lab signal tables directly into Delta from the committed `public/data/*.json`.
Tables created:
- `model_benchmarks` — Artificial Analysis model capability/pricing
- `hiring_jobs` — one row per job (tags + lifecycle + rich posting fields)
- `hiring_weekly` — per-company weekly totals / new / removed
- `hiring_weekly_breakdown` — per-company weekly composition by category / sub_area / vertical / theme

In [ ]:
# Paste the path to your repo's public/data folder
# (Workspace UI → right-click the `public/data` folder → Copy path)
DATA = "/Workspace/Users/aly.moosa@gatesfoundation.org/AI-Market-Intelligence/ai-market-intelligence-dashboard/public/data"

CATALOG = "fso_market_intelligence"
SCHEMA  = "frontier_labs"

In [ ]:
import json
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, BooleanType,
                                DoubleType, LongType, ArrayType)

## 1. Benchmarks → `model_benchmarks`

In [ ]:
models_schema = StructType([
    StructField("id", StringType()), StructField("name", StringType()),
    StructField("slug", StringType()), StructField("org", StringType()),
    StructField("country", StringType()), StructField("release_date", StringType()),
    StructField("open_weight", BooleanType()), StructField("param_count", LongType()),
    StructField("license", StringType()), StructField("modalities", ArrayType(StringType())),
    StructField("intelligence_index", DoubleType()), StructField("coding_index", DoubleType()),
    StructField("math_index", DoubleType()), StructField("gpqa", DoubleType()),
    StructField("hle", DoubleType()), StructField("mmlu_pro", DoubleType()),
    StructField("livecodebench", DoubleType()), StructField("ifbench", DoubleType()),
    StructField("lcr", DoubleType()), StructField("aime_25", DoubleType()),
    StructField("price_input", DoubleType()), StructField("price_output", DoubleType()),
    StructField("price_blended", DoubleType()), StructField("tokens_per_sec", DoubleType()),
    StructField("ttft", DoubleType()),
])

with open(f"{DATA}/models.json") as f:
    models = json.load(f)["models"]

bench = (
    spark.createDataFrame(models, schema=models_schema)
    .withColumn("release_date", F.to_date("release_date"))
    .withColumn("captured_at", F.current_date())
)
(bench.write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.model_benchmarks"))

print("model_benchmarks rows:", spark.table(f"{CATALOG}.{SCHEMA}.model_benchmarks").count())
display(spark.table(f"{CATALOG}.{SCHEMA}.model_benchmarks").limit(5))

## 2. Hiring → `hiring_jobs` (ledger tags/lifecycle + rich posting fields)
Joins `job_ledger.json` (all jobs incl. closed) with `jobs.json` (title, location, url, description for currently-listed jobs).
`description` is included for now — if Genie gets noisy, just add `.drop("description")`.

In [ ]:
# Ledger: classification + lifecycle
hiring_schema = StructType([
    StructField("job_id", StringType()), StructField("first_seen", StringType()),
    StructField("last_seen", StringType()), StructField("company", StringType()),
    StructField("category", StringType()), StructField("sub_area", StringType()),
    StructField("theme", StringType()), StructField("vertical", StringType()),
    StructField("social_impact", BooleanType()), StructField("active", BooleanType()),
])
with open(f"{DATA}/job_ledger.json") as f:
    led = json.load(f)
ledger_rows = [{"job_id": k, **v} for k, v in led.items()]
ledger_df = (
    spark.createDataFrame(ledger_rows, schema=hiring_schema)
    .withColumn("first_seen", F.to_date("first_seen"))
    .withColumn("last_seen",  F.to_date("last_seen"))
)

# Postings: rich fields (title, department, location, url, source, description)
posts_schema = StructType([
    StructField("id", StringType()), StructField("company", StringType()),
    StructField("title", StringType()), StructField("department", StringType()),
    StructField("location", StringType()), StructField("url", StringType()),
    StructField("source", StringType()), StructField("description", StringType()),
])
with open(f"{DATA}/jobs.json") as f:
    posts = json.load(f)["jobs"]
posts_df = (
    spark.createDataFrame(posts, schema=posts_schema)
    .withColumnRenamed("id", "job_id").drop("company")   # keep ledger's company
)

hiring = (
    ledger_df.join(posts_df, "job_id", "left")
    .withColumn("captured_at", F.current_date())
)
(hiring.write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.hiring_jobs"))

print("hiring_jobs rows:", spark.table(f"{CATALOG}.{SCHEMA}.hiring_jobs").count())
display(spark.table(f"{CATALOG}.{SCHEMA}.hiring_jobs").limit(5))

## 3. Weekly trends → `hiring_weekly` + `hiring_weekly_breakdown`

In [ ]:
with open(f"{DATA}/weekly_trends.json") as f:
    weeks = json.load(f)["weeks"]

# 3a. Per-company weekly summary
wk_rows = []
for wk in weeks:
    for company, d in wk["by_company"].items():
        wk_rows.append({
            "week": wk["week"], "company": company, "baseline": bool(wk.get("baseline", False)),
            "total": d.get("total"), "new": d.get("new"), "removed": d.get("removed"),
            "social_impact": d.get("social_impact"), "new_social_impact": d.get("new_social_impact"),
        })
weekly_schema = StructType([
    StructField("week", StringType()), StructField("company", StringType()),
    StructField("baseline", BooleanType()), StructField("total", LongType()),
    StructField("new", LongType()), StructField("removed", LongType()),
    StructField("social_impact", LongType()), StructField("new_social_impact", LongType()),
])
weekly = spark.createDataFrame(wk_rows, schema=weekly_schema).withColumn("week", F.to_date("week"))
(weekly.write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.hiring_weekly"))

# 3b. Per-company weekly composition (long format: one row per dimension value)
br_rows = []
for wk in weeks:
    for company, d in wk["by_company"].items():
        for dim, kv in (d.get("dist") or {}).items():        # category / sub_area / vertical / theme
            for value, count in kv.items():
                br_rows.append({"week": wk["week"], "company": company,
                                "dimension": dim, "value": value, "count": count})
br_schema = StructType([
    StructField("week", StringType()), StructField("company", StringType()),
    StructField("dimension", StringType()), StructField("value", StringType()),
    StructField("count", LongType()),
])
brk = spark.createDataFrame(br_rows, schema=br_schema).withColumn("week", F.to_date("week"))
(brk.write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.hiring_weekly_breakdown"))

print("hiring_weekly rows:", spark.table(f"{CATALOG}.{SCHEMA}.hiring_weekly").count())
print("hiring_weekly_breakdown rows:", spark.table(f"{CATALOG}.{SCHEMA}.hiring_weekly_breakdown").count())
display(spark.table(f"{CATALOG}.{SCHEMA}.hiring_weekly").limit(5))

## Going forward
- **Benchmarks:** daily job runs the API pull → append with `captured_at` (history) or overwrite (current).
- **Hiring:** scraper `MERGE`s into `hiring_jobs` on `job_id`; weekly tables recomputed each run.
- If `description` makes Genie noisy, add `.drop("description")` before the `hiring` write.